# Exploring Weights & Biases (W&B)

Use a Python 3 notebook; a GPU is helpful but optional. In Colab, run cells in order. Locally, open the notebook from its folder and install a compatible PyTorch/torchvision pair first. The setup installs `wandb` if missing and obtains `architectures.py` from the FA26 public repository if it is not available locally. The data cell downloads CIFAR-10.

The runnable example defaults to **offline mode**, which logs locally without an account or login. Offline logs do not appear in the hosted web dashboard until you sync them. For the dashboard portion, create/sign in to your [W&B account](https://wandb.ai/), set `WANDB_MODE = "online"`, and use the interactive `wandb.login()` prompt. Do not paste an API key into code or saved outputs. Alternatively, after logging in, sync an existing offline run with `wandb sync PATH_TO_OFFLINE_RUN` in a terminal (or `!wandb sync PATH_TO_OFFLINE_RUN` in a notebook), using the specific directory printed by W&B.

**Task:** run **ten distinct hyperparameter configurations**, choose the highest final **validation** accuracy within your search, and report that configuration and score. Complete `run()` and the configuration list, then enable `RUN_EXPERIMENTS`. Inspect the project run table and plots in the web dashboard for part (b); no graph submission is required. The two-batch example checks execution and does not count as a completed experiment. Keep the epoch budget fixed across configurations.


In [ ]:
# Install only the logging package if it is missing. Colab supplies PyTorch.
import importlib.util
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

if importlib.util.find_spec("wandb") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "wandb"])

# Local notebooks use their neighboring helper. A standalone Colab notebook fetches it.
candidates = [Path("architectures.py"), Path("../code/architectures.py")]
helper = next((p for p in candidates if p.is_file()), Path("architectures.py"))
if not helper.is_file():
    urlretrieve(
        "https://raw.githubusercontent.com/Berkeley-CS182/cs182fa26_public/main/hw06/code/architectures.py",
        helper,
    )
spec = importlib.util.spec_from_file_location("hw06_architectures", helper)
architectures = importlib.util.module_from_spec(spec)
spec.loader.exec_module(architectures)
BasicConvNet, ResNet18, MLP = architectures.BasicConvNet, architectures.ResNet18, architectures.MLP

import random
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 182
DATA_ROOT = "./data"
NUM_WORKERS = 0  # portable on Colab/Windows and makes sampling straightforward
print("PyTorch:", torch.__version__, "device:", device)


The supplied helpers train a model and evaluate it on a **fixed validation split** drawn from the CIFAR-10 training set. Use validation accuracy to choose hyperparameters. The official test set is not a tuning set. One epoch means one pass over the training split; keep the same number of epochs across configurations, even when batch size changes. Smaller batches produce more optimizer steps per epoch, which is part of the batch-size change.

Every run resets its seed; models with the same architecture start from the same parameters. A seed controls random choices but does not guarantee bit-for-bit agreement across hardware. Training metrics are sample-weighted averages across an epoch, while validation metrics are computed after the epoch in evaluation mode. Accuracy is recorded as a percentage.

`BasicConvNet` is a small CNN; `MLP` is a fully connected network that flattens the image internally. `ResNet18` is an optional, larger CNN with skip connections that add an earlier activation to a later block output. Its code is provided; deriving its architecture is not required. It is initialized without pretrained weights, accepts 32x32 images, and can be slower than the small CNN.


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
all_training_data = torchvision.datasets.CIFAR10(
    root=DATA_ROOT, train=True, download=True, transform=transform)
indices = torch.randperm(len(all_training_data),
                         generator=torch.Generator().manual_seed(SEED)).tolist()
split_at = int(0.9 * len(indices))
trainset = Subset(all_training_data, indices[:split_at])
valset = Subset(all_training_data, indices[split_at:])
print(f"Training: {len(trainset)}, validation: {len(valset)} examples")
# The official CIFAR-10 test set is reserved for a final evaluation after tuning.


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def prepare_run(config):
    seed_everything(config["seed"])
    model_factories = {"CNN": BasicConvNet, "MLP": MLP, "ResNet18": ResNet18}
    model = model_factories[config["architecture"]]().to(device)
    if config["optimizer"] == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=config["learning_rate"],
                                    momentum=config["momentum"])
    elif config["optimizer"] == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    else:
        raise ValueError("Choose SGD or Adam")
    trainloader = DataLoader(trainset, batch_size=config["batch_size"], shuffle=True,
                            num_workers=NUM_WORKERS,
                            generator=torch.Generator().manual_seed(config["seed"]))
    valloader = DataLoader(valset, batch_size=config["batch_size"], shuffle=False,
                          num_workers=NUM_WORKERS,
                          generator=torch.Generator().manual_seed(config["seed"]))
    return model, optimizer, trainloader, valloader

def train_one_epoch(model, optimizer, loader, max_batches=None):
    model.train()
    loss_sum, correct, count = 0.0, 0, 0
    for batch_index, (inputs, labels) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)  # each architecture handles flattening internally
        loss = nn.functional.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        count += labels.size(0)
    if count == 0:
        raise ValueError("No training examples were processed")
    return {"train/loss": loss_sum / count, "train/accuracy": 100 * correct / count}

@torch.no_grad()
def evaluate(model, loader, max_batches=None):
    model.eval()
    loss_sum, correct, count = 0.0, 0, 0
    for batch_index, (inputs, labels) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        inputs, labels = inputs.to(device), labels.to(device)
        logits = model(inputs)
        loss_sum += nn.functional.cross_entropy(logits, labels, reduction="sum").item()
        correct += (logits.argmax(dim=1) == labels).sum().item()
        count += labels.size(0)
    if count == 0:
        raise ValueError("No validation examples were processed")
    return {"val/loss": loss_sum / count, "val/accuracy": 100 * correct / count}

base_config = dict(learning_rate=0.01, batch_size=128, architecture="CNN",
                   optimizer="SGD", momentum=0.9, epochs=2, seed=SEED)


## Runnable logging example

The example logs one epoch on just two training and two validation batches.

In [ ]:
WANDB_MODE = "offline"  # Change to "online" only for your authenticated dashboard runs.
PROJECT = "cs182-hw06"
if WANDB_MODE == "online":
    wandb.login()  # interactive authentication; never put an API key in notebook code

def wandb_example(config, run_name="example", max_batches=2):
    model, optimizer, trainloader, valloader = prepare_run(config)
    # The context manager closes and flushes the run even when an exception occurs.
    with wandb.init(project=PROJECT, config=config, name=run_name,
                    mode=WANDB_MODE, dir=str(Path.cwd())) as tracking_run:
        for epoch in range(1, config["epochs"] + 1):
            metrics = train_one_epoch(model, optimizer, trainloader, max_batches)
            metrics.update(evaluate(model, valloader, max_batches))
            tracking_run.log({"epoch": epoch, **metrics}, step=epoch)
            print(f"{run_name}, epoch {epoch}: {metrics}")
        tracking_run.summary["final_val_accuracy"] = metrics["val/accuracy"]
        tracking_run.summary["final_val_loss"] = metrics["val/loss"]
        print("Run files:", tracking_run.dir)
    return metrics

example_config = dict(base_config, epochs=1)
example_metrics = wandb_example(example_config)
print("Example processed only two batches; do not report it as a full experiment.")


## Comparing the tools

In an online or synced W&B project, add your configuration values and `final_val_accuracy` as run-table columns, sort/filter runs, and compare their learning curves. Record what this workflow makes easier or harder for you.

Compare W&B's hosted project organization, collaboration, and experiment-management workflow with your TensorBoard experience. Both tools can visualize metrics and compare hyperparameters; TensorBoard's HParams dashboard also supports filtering and sorting. Avoid claiming these shared capabilities are exclusive to W&B. See [W&B's run documentation](https://docs.wandb.ai/ref/python/experiments/run/) for logging and offline/online behavior.


## Your experiments

Choose distinct configurations, use the full training/validation data, and record every changed setting. If you want to attribute an effect to one setting (an ablation), hold the other settings fixed. Report the best configuration within your actual search, not a guessed optimum.

In [ ]:
hyperparameters = []  # TODO: choose ten distinct configuration dictionaries.

def run(config, run_name, max_batches=None):
    # TODO: adapt the example to train, validate, and log one configuration.
    # Return a dictionary containing val/accuracy and val/loss.
    raise NotImplementedError("Implement your W&B experiment")

RUN_EXPERIMENTS = False  # Set True after completing run() and the ten configs.
if RUN_EXPERIMENTS:
    assert len(hyperparameters) == 10, "Choose ten configurations"
    experiment_results = [run(config, run_name=f"trial-{i}")
                          for i, config in enumerate(hyperparameters)]


The CIFAR training example builds on [PyTorch tutorials](https://github.com/pytorch/tutorials/), distributed under the [BSD 3-Clause license](https://github.com/pytorch/tutorials/blob/main/LICENSE).